In [1]:
import os
import re
import numpy as np
import pandas as pd
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")


PyTorch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
GPU Memory: 8.2 GB


In [ ]:

import transformers
import librosa
import whisper
import jiwer
from jiwer import wer

print(f"Transformers version: {transformers.__version__}")
print(f"Librosa version: {librosa.__version__}")
print(f"Whisper version: {whisper.__version__}")
print("Jiwer: OK")
print("All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.9 MB/s eta 0:00:00
Transformers version: 5.0.0
Librosa version: 0.11.0
Whisper version: 20250625
Jiwer: OK
All libraries installed successfully!


In [ ]:
BASE_PATH = "audio_data"
TRAIN_PATH = f"{BASE_PATH}/train"
TEST_PATH = f"{BASE_PATH}/test"

train_df = pd.read_csv(f'{BASE_PATH}/train.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test.csv')

In [4]:
train_df['audio_path'] = train_df['audio'].apply(lambda x: TRAIN_PATH + "/" + x)
test_df['audio_path'] = test_df['audio'].apply(lambda x: TEST_PATH + "/" + x)

train_exists = train_df['audio_path'].apply(os.path.exists).sum()
test_exists = test_df['audio_path'].apply(os.path.exists).sum()

print(f"Train files found: {train_exists} / {len(train_df)}")
print(f"Test files found: {test_exists} / {len(test_df)}")
print(f"\nSample train rows:")
display(train_df[['audio', 'audio_path', 'text']].head(3))

Train files found: 2000 / 2000
Test files found: 100 / 100

Sample train rows:


,audio,audio_path,text
0,audio_00000.wav,/kaggle/input/competitions/multilingual-speech...,you had quoted plutarch line.
1,audio_00001.wav,/kaggle/input/competitions/multilingual-speech...,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2,audio_00002.wav,/kaggle/input/competitions/multilingual-speech...,to do his phd in engineering about four years ...


In [ ]:
def detect_language(text):
    if not isinstance(text, str):
        return 'unknown'
    # Tamil script
    if re.search(r'[\u0B80-\u0BFF]', text):
        return 'Tamil'
    # Hindi/Devanagari script
    if re.search(r'[\u0900-\u097F]', text):
        return 'Hindi'
    # Default to English/Latin
    return 'English/Latin'

train_df['language'] = train_df['text'].apply(detect_language)

print("=== Language Distribution in Training Set ===")
print(train_df['language'].value_counts())
print(f"\nTotal samples: {len(train_df)}")


print("\n=== Sample per Language ===")
for lang in train_df['language'].unique():
    sample = train_df[train_df['language'] == lang].iloc[0]

    print(f"\n[{lang}] {sample['text'][:80]}")

=== Language Distribution in Training Set ===
language
English/Latin    998
Hindi            508
Tamil            494
Name: count, dtype: int64

Total samples: 2000

=== Sample per Language ===

[English/Latin] you had quoted plutarch line.

[Tamil] மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்ம மலை ஏறினோம் அப்டிங்கிறத விட அங்க

[Hindi] इस किताब में दर्ज कुल तेरह चैप्टरों में शामिल बीसियों महिलाओं की


In [6]:
train_df['text'] = train_df['text'].apply(lambda x: x.lower())
train_df

,id,audio,text,audio_path,language
0,0,audio_00000.wav,you had quoted plutarch line.,/kaggle/input/competitions/multilingual-speech...,English/Latin
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...,/kaggle/input/competitions/multilingual-speech...,Tamil
2,2,audio_00002.wav,to do his phd in engineering about four years ...,/kaggle/input/competitions/multilingual-speech...,English/Latin
3,3,audio_00003.wav,maybe he was not at home.,/kaggle/input/competitions/multilingual-speech...,English/Latin
4,4,audio_00004.wav,but we didn't break his old window you know ex...,/kaggle/input/competitions/multilingual-speech...,English/Latin
...,...,...,...,...,...
1995,1995,audio_01995.wav,நிறைய வகையான போட்டி நடக்கும் அப்புறம் வந்து உர...,/kaggle/input/competitions/multilingual-speech...,Tamil
1996,1996,audio_01996.wav,exclaimed the other as though more than surpri...,/kaggle/input/competitions/multilingual-speech...,English/Latin
1997,1997,audio_01997.wav,पाकिस्तान ने आईएमएफ़ से करीब आठ से बारह डॉलर ब...,/kaggle/input/competitions/multilingual-speech...,Hindi
1998,1998,audio_01998.wav,इसराइल की यात्रा की थी,/kaggle/input/competitions/multilingual-speech...,Hindi


In [7]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import datasets
from datasets import ClassLabel
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import gc

In [8]:
gc.collect()
torch.cuda.empty_cache()
print(f"GPU Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

GPU Free: 15.64 GB


In [9]:
MODEL_ID = "openai/whisper-large-v3-turbo"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model = model.to(torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16)
model = model.to(device)

model.config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [10]:
print(f"GPU Used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

GPU Used: 1.62 GB / 15.64 GB


In [11]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

In [12]:
def load_audio(audio_path, sampling_rate = 16000):
    audio, _ = librosa.load(audio_path, sr = sampling_rate)
    return audio


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    dtype=torch.bfloat16 if use_bf16 else torch.float16

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].to(self.dtype)
        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        batch["attention_mask"] = labels_batch.attention_mask 

        return batch

In [13]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor = processor,
    decoder_start_token_id= model.config.decoder_start_token_id
)

In [14]:
def prepare_data(batch):
    audio = load_audio(batch['audio_path'])
    batch['input_features'] = processor(audio, sampling_rate = 16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

In [15]:
train_data = datasets.Dataset.from_pandas(train_df[['audio_path', 'text', 'language']])
languages = train_data.unique("language")
train_data = train_data.cast_column("language", ClassLabel(names=languages))

split = train_data.train_test_split(test_size=0.05, seed=42, stratify_by_column="language")
train_split = split["train"]
eval_split  = split["test"]

train_split = train_split.remove_columns("language")
eval_split  = eval_split.remove_columns("language")

train_split = train_split.map(prepare_data, remove_columns=train_split.column_names)
eval_split  = eval_split.map(prepare_data, remove_columns=eval_split.column_names)

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [16]:
print(f"Train samples after mapping: {len(train_split)}")
print(f"Eval samples after mapping: {len(eval_split)}")

Train samples after mapping: 1900
Eval samples after mapping: 100


In [17]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate

In [18]:
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer_score = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer_score}


training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper-large-finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    dataloader_num_workers=4,

    learning_rate=3e-5,
    warmup_steps=50,
    num_train_epochs=5,
    weight_decay=0.01,

    gradient_checkpointing=True,
    fp16=use_fp16,
    bf16=use_bf16,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    predict_with_generate=True,
    generation_max_length=444,

    logging_steps=10,
    report_to="none",
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=eval_split,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics
)

In [19]:
print("Starting fine-tuning...")
print(f"Total epochs: 5 | Batch size: 4 | Grad accum: 2")
print(f"Total samples: {len(train_split)}")
print("="*50)

trainer.train()

Starting fine-tuning...
Total epochs: 5 | Batch size: 4 | Grad accum: 2
Total samples: 1900


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
50,1.672070,0.472506,0.316400
100,1.548828,0.407452,0.240751
150,1.013623,0.369366,0.220872
200,0.975098,0.352915,0.215903
250,0.597070,0.336839,0.200442
300,0.640381,0.336013,0.205964
350,0.528516,0.319261,0.195472
400,0.443182,0.342773,0.213142
450,0.398120,0.322566,0.197681
500,0.230884,0.330153,0.202098


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=595, training_loss=0.8523354923023897, metrics={'train_runtime': 25295.5053, 'train_samples_per_second': 0.376, 'train_steps_per_second': 0.024, 'total_flos': 1.619703595008e+19, 'train_loss': 0.8523354923023897, 'epoch': 5.0})

In [ ]:
trainer.save_model("/models/whisper-finetuned/best_model")
processor.save_pretrained("/models/whisper-finetuned/best_model")

print("Best model and processed data saved!")

# Verify saved files
import os
for path in [
    "/models/whisper-finetuned/best_model",
]:
    files = os.listdir(path)
    print(f"\n{path}:\n", files)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model and processed data saved!

/kaggle/working/whisper-finetuned/best_model:
 ['config.json', 'tokenizer.json', 'training_args.bin', 'model.safetensors', 'processor_config.json', 'preprocessor_config.json', 'generation_config.json', 'tokenizer_config.json']


In [ ]:
import torch
import librosa
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the fine-tuned model
MODEL_DIR = "/models/working/whisper-finetuned/best_model"
print("Loading fine-tuned model...")
processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model_inf = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR)
model_inf = model_inf.to(device)
model_inf.eval()
print(f"Model loaded! GPU: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

# Load test CSV
BASE_PATH = "audio"
TEST_AUDIO_PATH = f"{BASE_PATH}/test"
test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
test_df['audio_path'] = test_df['audio'].apply(lambda x: os.path.join(TEST_AUDIO_PATH, x))

print(f"\nGenerating predictions for {len(test_df)} test files...")
print("="*50)

predictions = []
dtype = next(model_inf.parameters()).dtype
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        try:
            # Load audio
            audio, _ = librosa.load(row['audio_path'], sr=16000)

            # Process
            input_features = processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            ).to(dtype).input_features.to(device)

            # Generate transcription
            predicted_ids = model_inf.generate(
                input_features,
                max_new_tokens=444,
                num_beams=5
            )

            # Decode
            transcription = processor.batch_decode(
                predicted_ids,
                skip_special_tokens=True
            )[0].strip()

            predictions.append(transcription)

        except Exception as e:
            print(f"Error on {row['audio']}: {e}")
            predictions.append("")


print("\nCreating submission file...")
output_df = pd.DataFrame({
    'audio': test_df['audio'],
    'text': predictions
})

output_df.to_csv("prediction.csv", index=False)
print("Submission saved to prediction.csv")
print(f"\nSample predictions:")
for i in range(5):
    print(f"[{i}] {output_df['audio'].iloc[i]} → {output_df['text'].iloc[i][:80]}")

Loading fine-tuned model...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Model loaded! GPU: 6.51 GB

Generating predictions for 100 test files...


  0%|          | 0/100 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 100/100 [06:07<00:00,  3.67s/it]


Creating submission file...
Submission saved to /kaggle/working/submission.csv

Sample predictions:
[0] audio_00000.wav → इनमें जेपी मॉर्गन सिम्मस और ब्लेक रोग के सीईयों अमिल हैं
[1] audio_00001.wav → एक हज़ार नौ सौ नबे में जहां दो मुस्लिम उम्मीद्वानों ने जीत
[2] audio_00002.wav → மாட்ட வந்து மாட்டுப்பொங்கள் முதிஞ்சி மாட்டுப்பொங்கள் நைட்டு வந்து மாட வந்து மாட்
[3] audio_00003.wav → அருவிக்கிற மாதிரி ஒரு படை இருக்கணும்
[4] audio_00004.wav → and by this i knew the violence of my love i left the table without a moment's h
